# PAS: detector YOLO11 (caixas) no Kaggle

Treina detectores **YOLO11** (Ultralytics) para marcar os pólipos com caixas, em vez de segmentar.

- **Dados:** as mesmas imagens e o mesmo split treino/validação do EMCAD (Kvasir-SEG + CVC-ClinicDB).
- **Caixas de gabarito:** vêm das máscaras.
- **Avaliação:** nos 5 conjuntos de teste, com `scripts/eval_boxes.py`. É a mesma métrica usada para as caixas tiradas da segmentação, então a comparação é direta.

**Antes de rodar:** em *Settings*, coloque *Accelerator = GPU T4 x2* e *Internet = On*.

> Licença: o Ultralytics é **AGPL-3.0**. Para pesquisa não há problema; num produto fechado, é preciso licença comercial ou trocar o detector.

In [ ]:
MODELS = ["yolo11n.pt", "yolo11s.pt"]   # n: 2,6M params | s: 9,4M params
EPOCHS = 100
IMGSZ = 640
BATCH = 16
BRANCH = "claude/gifted-albattani-ikhwce"
REPO = "warrior-graph/PAS-PolypAutoSegmentation"

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Sem GPU: Settings -> Accelerator -> GPU T4 x2 (exige telefone verificado)"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
import os, shutil, subprocess, urllib.error, urllib.request

DEST = "/kaggle/working/PAS"
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass


def git(*args, cwd=None):
    """Run git, never echoing the token in errors."""
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode:
        msg = (r.stdout + r.stderr).replace(token, "***") if token else r.stdout + r.stderr
        hints = {
            "Could not resolve host": "Sem Internet: ative Settings -> Internet: On.",
            "could not read Username": "Sem token: anexe o secret GITHUB_TOKEN a este notebook (Add-ons -> Secrets) ou torne o repo público.",
            "Repository not found": "Token sem acesso ao repo (resource owner, Contents: Read, aprovação da org).",
            "Authentication failed": "Token inválido/expirado ou sem acesso ao repo.",
            "not found in upstream": "Branch inexistente: confira BRANCH.",
            "couldn't find remote ref": "Branch inexistente: confira BRANCH.",
        }
        hint = next((h for k, h in hints.items() if k in msg), "")
        raise RuntimeError(f"git {args[0]} falhou:\n{msg}\n{hint}") from None
    return r.stdout


try:
    urllib.request.urlopen("https://github.com", timeout=10)
except urllib.error.HTTPError:
    pass  # respondeu: há Internet
except Exception:
    raise RuntimeError("Sem acesso à Internet: ative Settings -> Internet: On (exige telefone verificado).") from None

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
try:
    if os.path.isdir(os.path.join(DEST, ".git")):
        # Clone anterior: atualiza para a última versão do branch
        git("remote", "set-url", "origin", url, cwd=DEST)
        git("fetch", "--depth", "1", "origin", BRANCH, cwd=DEST)
        git("reset", "--hard", "FETCH_HEAD", cwd=DEST)
    else:
        shutil.rmtree(DEST, ignore_errors=True)  # resto de um clone que falhou
        git("clone", "-b", BRANCH, "--depth", "1", url, DEST)
finally:
    # /kaggle/working vai para o Output: nunca deixar o token no .git/config
    if os.path.isdir(os.path.join(DEST, ".git")):
        subprocess.run(["git", "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], cwd=DEST)
print(git("log", "--oneline", "-1", cwd=DEST))
os.chdir(DEST)


def run(cmd):
    """Run a shell command streaming its output; fail the cell (and a Save & Run All) on error."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"comando falhou ({p.returncode}): {cmd}")

In [ ]:
run("pip install -q -U ultralytics timm 'albumentations>=2.0' gdown")

In [ ]:
DATA = "/tmp/pas_data"
run(f"python scripts/download_data.py --root {DATA}")
run(f"python scripts/make_yolo_dataset.py --data-root {DATA} --out /tmp/yolo_data")

In [ ]:
PROJECT = os.path.join(os.getcwd(), "runs", "yolo")
for m in MODELS:
    name = os.path.splitext(m)[0]
    run(f"yolo detect train data=/tmp/yolo_data/data.yaml model={m} epochs={EPOCHS} imgsz={IMGSZ} "
        f"batch={BATCH} seed=42 workers=4 project={PROJECT} name={name} exist_ok=True")

In [ ]:
# Mesma métrica das caixas tiradas da segmentação (scripts/eval_boxes.py)
from IPython.display import Markdown, display
for m in MODELS:
    name = os.path.splitext(m)[0]
    run(f"python scripts/eval_boxes.py --yolo {PROJECT}/{name}/weights/best.pt --imgsz {IMGSZ} "
        f"--data-root {DATA} --out {PROJECT}/{name}/boxes")
    display(Markdown(f"### {name}\n\n" + open(f"{PROJECT}/{name}/boxes.md").read()))

In [ ]:
# Empacota pesos, curvas e métricas (aba Output)
import glob, zipfile
with zipfile.ZipFile("/kaggle/working/yolo_run.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for m in MODELS:
        name = os.path.splitext(m)[0]
        d = f"{PROJECT}/{name}"
        files = [f"{d}/weights/best.pt", f"{d}/results.csv", f"{d}/args.yaml"] + glob.glob(f"{d}/boxes*") + glob.glob(f"{d}/*.png")
        for f in files:
            if os.path.exists(f):
                z.write(f, os.path.relpath(f, PROJECT))
run("ls -lh /kaggle/working/yolo_run.zip")